In [1]:
#!/usr/bin/env python
# coding: utf-8

import json
import argparse
import os
import tqdm
import pandas as pd
import numpy as np

import base64
import mimetypes

import anthropic

def encode_image(image_path):
    with open(image_path, "rb") as f:
        data = base64.b64encode(f.read()).decode("utf-8")
    media_type = mimetypes.guess_type(image_path)[0] or "application/octet-stream"
    return data, media_type

In [ ]:
def load_template(template_dir, template_name):
    template_path = os.path.join(template_dir, f'prompt-{template_name}.txt')
    if not os.path.exists(template_path):
        raise FileNotFoundError(f"Template not found: {template_path}")
    with open(template_path, 'r', encoding='utf-8') as f:
        template_text = f.read()
    return template_text

In [3]:
def get_instruction_text(template_dir, base_template_name, story_images, character_images=None, character_names=None):
    if character_images and len(character_images) > 0:
        template_name = f"{base_template_name}-w-names"
    else:
        template_name = f"{base_template_name}-wo-names"

    template = load_template(template_dir, template_name)

    fill_values = {
        'story_images': story_images,
        'num_story_images': len(story_images)
    }

    if character_images:
        fill_values['character_images'] = character_images
        fill_values['num_character_images'] = len(character_images)
        fill_values['character_images_text'] = (
            '1 character image' if len(character_images) == 1 else f"{len(character_images)} character images"
        )
    else:
        fill_values['character_images'] = []
        fill_values['num_character_images'] = 0
        fill_values['character_images_text'] = 'no character images'

    if character_names:
        fill_values['character_names'] = character_names
        fill_values['character_names_text'] = ', '.join(character_names)
    else:
        fill_values['character_names'] = []
        fill_values['character_names_text'] = 'no character names'

    filled_instruction = template.format(**fill_values)

    #print('FILLED INSTRUCTION', filled_instruction)
    
    return filled_instruction

In [ ]:
def split_dataframe(df, random_seed=42):
    np.random.seed(random_seed)
    
    grouped = df.groupby('story_id')
    
    df1_rows = []
    df2_rows = []
    df3_rows = []
    
    for story_id, group in grouped:
        assert len(group) == 3, f"Story ID {story_id} does not have exactly 3 rows."
        
        group = group.sample(frac=1, random_state=random_seed)
        
        df1_rows.append(group.iloc[0])
        df2_rows.append(group.iloc[1])
        df3_rows.append(group.iloc[2])
    
    return pd.DataFrame(df1_rows).reset_index(drop=True), pd.DataFrame(df2_rows).reset_index(drop=True), pd.DataFrame(df3_rows).reset_index(drop=True)

In [ ]:
def run(df,
        output_dir,
        prompt_dir,
        prompt_name,
        model,
        client,
        seed):

    if prompt_name == "large":
        output_dir = os.path.join(output_dir, f"seed-{seed}")
    os.makedirs(output_dir, exist_ok=True)
    
    if prompt_name == "large":
        system_prompt = (
            "You are a helpful assistant and an experienced expert crowdworker. "
            "You are qualified to perform the following task. The title of the task you are working on is: "
            "\"Help us bridge the gap between AI and humans in telling stories about movies!\" "
            "The task description is as follows: we are a group of researchers working with large language models, "
            "and we ask for your help in collecting stories based on the images provided. The data you submit will "
            "be used to build and improve AI models that understand how to generate stories about movies just as you do! "
            "We're very excited to have you join our experiment! Please carefully read the instructions. "
            "You must follow all instructions in order to be eligible for payment."
        )
    else:
        system_prompt = (
            "You are a helpful assistant and an experienced expert crowdworker. "
            "You are qualified to perform the following task."
        )


    for idx, row in tqdm.tqdm(df.iterrows(), total=len(df)):
        
        story_id = row['story_id']

        # If the output file already exists, skip this story
        output_file = os.path.join(output_dir, f"{story_id}.parquet")
        if os.path.exists(output_file):
            tqdm.tqdm.write(f"Skipping story_id {story_id}: output already exists at {output_file}")
            continue

        basepath_images = '../../data/sampled_60/images/'
        basepath_character_images = '../../data/sampled_60/characters/'

        # Construct story image paths based on story_id
        story_images = []
        for i in range(10):  # Check for img0 through img9
            img_path = f'{basepath_images}{story_id}_img{i}.jpg'
            if os.path.exists(img_path):
                story_images.append(img_path)
        
        # Construct character image paths and extract character names
        character_images = []
        character_names = []
        for i in range(5):  # Check for char0 through char4
            char_col = f'char{i}'
            char_path = f'{basepath_character_images}{story_id}_char{i}.jpg'
            
            # Check if character name exists and is not empty
            if char_col in row and pd.notna(row[char_col]) and row[char_col] not in [None, '', '{}']:
                character_names.append(row[char_col])
                
                # Check if corresponding character image file exists
                if os.path.exists(char_path):
                    character_images.append(char_path)
        
        # Convert to None if empty
        character_images = character_images if len(character_images) > 0 else None
        character_names = character_names if len(character_names) > 0 else None

        if character_images is not None:
            all_images = story_images + character_images
        else:
            all_images = story_images

        print('all images', all_images)


        instruction_text = get_instruction_text(
            prompt_dir,
            base_template_name=prompt_name,
            story_images=story_images,
            character_images=character_images,
            character_names=character_names
        )

        content = []
        
        for idx, path in enumerate(all_images, start=1):
            image_data, media_type = encode_image(path)
            content.extend([
                {
                    "type": "text",
                    "text": f"Image {idx}:"
                },
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": media_type,
                        "data": image_data,
                    },
                }
            ])


        content.append({
            "type": "text",
            "text": instruction_text
        })
        
        message = {
            "model": model_path,
            # change tokens, improve code functionality here also
            "max_tokens": 4096,
            "temperature": 0.6,
            "system": system_prompt,
            "messages": [
                {
                    "role": "user",
                    "content": content
                }
            ],
        } 

        try:

            if prompt_name == "large" and seed == 42:

                # SPEAKER 1
                response = client.messages.create(**message)
            
            if prompt_name == "large" and seed == 43:

                # SPEAKER 2
                response = client.messages.create(**message)

            if prompt_name == "large" and seed == 44:

                # SPEAKER 3
                response = client.messages.create(**message)

            else:

                # medium or original prompts, no "multiple speakers"
                response = client.messages.create(**message)

            model_out = response.content
            combined_text = '\n'.join(block.text for block in model_out if block.type == "text")

        except Exception as e:
            tqdm.tqdm.write(f'Failed on story_id {story_id}: {e}')
            continue

        output_file = os.path.join(output_dir, f"{story_id}.parquet")

        #print(combined_text)
        result_df = pd.DataFrame([{ 
            "story_id": story_id,
            "instruction_text": instruction_text,
            "model_output": combined_text,
            "model": model,
            "seed": seed
        }])

        result_df.to_parquet(output_file, index=False)
        
        tqdm.tqdm.write(f"Saved output for story_id {story_id} -> {output_file}")

In [ ]:
input_file = '../../data/sampled_60/sampled_60_stories.json'
output_dir = './out-claude45-60stories/'
template_dir = '../../data/prompts/'
model_path = 'claude-sonnet-4-5-20250929'

template_name = 'large'

In [7]:
df = pd.read_json(input_file, lines=True)
print(df.shape)

rep_df = df.groupby('story_id').first().reset_index()
rep_df = rep_df.loc[rep_df.index.repeat(3)].reset_index(drop=True)
assert all(rep_df.groupby('story_id').size() == 3), 'Not all story_id have 3 repetitions'
df = rep_df
print('Expanded df shape ->', df.shape)

(60, 41)
Expanded df shape -> (180, 41)


In [8]:
df.shape

(180, 41)

In [10]:
import numpy as np

df1, df2, df3 = split_dataframe(df, random_seed=42)

seeds = [42] if template_name in ['original'] else [42, 43, 44]

In [11]:
df1.shape

(60, 41)

In [12]:
seeds

[42, 43, 44]

In [ ]:
final_output_dir = os.path.join(output_dir, f'prompt-{template_name}-outputs')
os.makedirs(final_output_dir, exist_ok=True)

# Load API key from config file
with open('config.json', 'r') as f:
    config = json.load(f)

client = anthropic.Anthropic(
    api_key=config['anthropic_api_key'],
)

if template_name in ['original']:
    print(f"Running with template: {template_name}, only df1, seed {seeds[0]}")
    #print(df1.shape)

    run(df1,
        final_output_dir,
        template_dir,
        template_name,
        model_path,
        client,
        seed=seeds[0]
       )


elif template_name == 'large':
    print(f"Running with template: {template_name}, df1, df2, df3 separately")

    for num, single_df in enumerate([df1, df2, df3]):
        #print(single_df.shape)
        current_seed = seeds[num]

        run(single_df,
            final_output_dir,
            template_dir,
            template_name,
            model_path,
            client,
            seed=current_seed
           )

Running with template: large, df1, df2, df3 separately


100%|██████████| 60/60 [00:00<00:00, 757.91it/s]


Skipping story_id 345: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-42/345.parquet
Skipping story_id 515: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-42/515.parquet
Skipping story_id 721: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-42/721.parquet
Skipping story_id 723: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-42/723.parquet
Skipping story_id 733: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-42/733.parquet
Skipping story_id 948: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-42/948.parquet
Skipping story_id 1111: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-42/1111.parquet
Skipping story_id 1201: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-42/1201.parquet
Skipping story_id 1214: output already exists at ./out-claude45-60stories/prompt-lar

100%|██████████| 60/60 [00:00<00:00, 675.77it/s]


Skipping story_id 345: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-43/345.parquet
Skipping story_id 515: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-43/515.parquet
Skipping story_id 721: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-43/721.parquet
Skipping story_id 723: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-43/723.parquet
Skipping story_id 733: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-43/733.parquet
Skipping story_id 948: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-43/948.parquet
Skipping story_id 1111: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-43/1111.parquet
Skipping story_id 1201: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-43/1201.parquet
Skipping story_id 1214: output already exists at ./out-claude45-60stories/prompt-lar

...

Skipping story_id 345: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/345.parquet
Skipping story_id 515: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/515.parquet
Skipping story_id 721: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/721.parquet
Skipping story_id 723: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/723.parquet
Skipping story_id 733: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/733.parquet
Skipping story_id 948: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/948.parquet
Skipping story_id 1111: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/1111.parquet
Skipping story_id 1201: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/1201.parquet
Skipping story_id 1214: output already exists at ./out-claude45-60stories/prompt-lar

  0%|          | 0/60 [00:00<?, ?it/s]

Skipping story_id 4047: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/4047.parquet
Skipping story_id 4173: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/4173.parquet


100%|██████████| 60/60 [00:00<00:00, 641.23it/s]

Skipping story_id 4797: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/4797.parquet
Skipping story_id 5047: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/5047.parquet
Skipping story_id 5062: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/5062.parquet
Skipping story_id 5201: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/5201.parquet
Skipping story_id 5265: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/5265.parquet
Skipping story_id 5349: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/5349.parquet
Skipping story_id 5540: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/5540.parquet
Skipping story_id 5721: output already exists at ./out-claude45-60stories/prompt-large-outputs/seed-44/5721.parquet
Skipping story_id 6111: output already exists at ./out-claude45-60storie